# 🏥 ICU Sepsis Prediction — End-to-End ML Pipeline

> **Dataset**: Synthetic MIMIC-IV style ICU data (5,000 patients, 77 features)  
> **Task**: Binary classification — predict sepsis onset within 24 hours  
> **Target**: `sepsis_label` (1 = Sepsis, 0 = No sepsis) | Class balance: ~15% positive

---

## 📋 Notebook Roadmap

| Step | Section | Key Skills |
|------|---------|------------|
| 1 | Environment Setup | Libraries, config |
| 2 | Data Loading & First Look | Shape, dtypes, nulls |
| 3 | Data Quality Audit | Outliers, artifacts, impossible values |
| 4 | Exploratory Data Analysis | Distributions, correlations, clinical patterns |
| 5 | Data Cleaning | Fix errors, unit conversion, typos |
| 6 | Feature Engineering | Derived features, interactions, flags |
| 7 | Preprocessing Pipeline | Imputation, encoding, scaling |
| 8 | Model Training | XGBoost |
| 9 | Evaluation & Comparison | ROC-AUC, PR-AUC, SHAP explainability |
| 10 | Threshold Tuning | Clinical cost-sensitive optimization |
| 11 | Final Summary | Key findings |
| 12 | Export to .pkl | For reading python file |

---

## Step 1 — Environment Setup

We install and import every library we'll need upfront. This keeps later cells clean and makes dependency issues visible immediately.

In [ ]:
# Install any packages not available on Kaggle by default
!pip install -q lightgbm shap imbalanced-learn

In [ ]:
# ── Core ─────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.3f}'.format)

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PALETTE = {'0': '#4C72B0', '1': '#DD8452'}
FIG_SIZE = (14, 5)

# ── Preprocessing ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA

# ── Imbalance ─────────────────────────────────────────────────────────────────
from imblearn.pipeline import Pipeline as ImbPipeline

# ── Models ────────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb
import lightgbm as lgb

# ── Evaluation ────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay,
    roc_curve, precision_recall_curve, f1_score, brier_score_loss
)

# ── Explainability ────────────────────────────────────────────────────────────
import shap

# ── Special Colab ────────────────────────────────────────────────────────────
from google.colab import files

print('All libraries loaded')
print(f'NumPy: {np.__version__}  |  Pandas: {pd.__version__}  |  XGBoost: {xgb.__version__}  |  LGB: {lgb.__version__}')

In [ ]:
# ── Global config ─────────────────────────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE     = 0.20   # 20% held-out test set
N_FOLDS       = 5      # StratifiedKFold CV
TARGET        = 'sepsis_label'
DROP_COLS     = ['subject_id']   # identifiers — never use as features

np.random.seed(RANDOM_STATE)

---
## Step 2 — Data Loading & First Look

We do a structured first inspection covering:
- Shape and column inventory  
- Dtype audit (numeric vs object)  
- Missing value map  
- Class distribution

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
# Kaggle path — adjust if running locally
DATA_PATH = '/sepsis_icu_synthetic.csv'

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
# ── Column inventory ──────────────────────────────────────────────────────────
col_groups = {
    'Demographics': ['subject_id','age','gender','weight_kg','height_cm','bmi','ethnicity','insurance'],
    'Heart Rate':   [c for c in df_raw.columns if c.startswith('hr_')],
    'Blood Pressure': [c for c in df_raw.columns if c.startswith('sbp_') or c.startswith('dbp_') or c == 'map_mean'],
    'Temperature':  [c for c in df_raw.columns if c.startswith('temp_')],
    'SpO2':         [c for c in df_raw.columns if c.startswith('spo2_')],
    'Resp Rate':    [c for c in df_raw.columns if c.startswith('respiratory_')],
    'Labs':         ['wbc','lactate_mmol','creatinine','platelet_count','bilirubin_total',
                     'glucose','ph_arterial','pao2_fio2_ratio','inr','sodium','potassium',
                     'chloride','bicarbonate','hematocrit','hemoglobin'],
    'Comorbidities': ['diabetes','hypertension','chf','copd','chronic_kidney_disease',
                      'liver_disease','immunosuppression','cad','atrial_fibrillation','cancer_active'],
    'Interventions': ['vasopressors_flag','mechanical_ventilation','fio2_percent','antibiotics_24h',
                      'fluids_ml_24h','sedation_score','vasopressor_dose_mcg_kg_min','insulin_infusion_flag'],
    'Scores':        ['sofa_score','apache_iv','qsofa','sirs_criteria','gcs_total'],
    'Admin':         ['icu_los_hours','hospital_admit_source','icu_admit_time_hour','day_of_week','readmission_30day'],
}

for grp, cols in col_groups.items():
    print(f'{grp:20s} → {len(cols):2d} cols')
print(f'\nTarget → {TARGET}')

In [ ]:
# ── Dtype audit ───────────────────────────────────────────────────────────────
print('Object columns (potential mixed-type issues):')
for col in df_raw.select_dtypes('object').columns:
    n_unique = df_raw[col].nunique()
    sample   = df_raw[col].dropna().unique()[:5].tolist()
    print(f'{col:35s} nunique={n_unique:4d}  sample={sample}')

In [ ]:
# ── Missing value heatmap ─────────────────────────────────────────────────────
miss_pct = df_raw.isnull().mean().sort_values(ascending=False)
miss_cols = miss_pct[miss_pct > 0]

fig, ax = plt.subplots(figsize=(14, 4))
miss_cols.plot(kind='bar', ax=ax, color='#DD8452', edgecolor='white')
ax.axhline(0.10, ls='--', color='red', linewidth=1, label='10% threshold')
ax.set_title('Missing Data by Column', fontsize=14, fontweight='bold')
ax.set_ylabel('Missing Fraction')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f'\nColumns with >10% missing: {(miss_pct > 0.10).sum()}')
print(f'Overall missingness: {df_raw.isnull().mean().mean():.3f}')

In [ ]:
# ── Target class distribution ─────────────────────────────────────────────────
counts = df_raw[TARGET].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['No Sepsis (0)', 'Sepsis (1)'], counts.values,
            color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=1.2)
axes[0].set_title('Class Distribution (Counts)', fontweight='bold')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['No Sepsis', 'Sepsis'],
            colors=['#4C72B0','#DD8452'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Class Distribution (%)', fontweight='bold')

plt.tight_layout()
plt.show()
print(f'Imbalance ratio: 1:{counts[0]//counts[1]} — requires class_weight handling')

---
## Step 3 — Data Quality Audit

Real EHR data is messy. We systematically hunt for:
- **Physiologically impossible values** (temp = 50°C, HR = 400)
- **Unit confusion errors** (weight in lbs entered as kg)
- **Categorical typos** ("Mael" instead of "Male")
- **Blood pressure swap errors** (DBP > SBP)
- **Documentation paradoxes** (GCS=15 but mechanically ventilated)
- **Mixed-type columns** (WBC stored as string with ">" prefix)

In [ ]:
df = df_raw.copy()

issues = {}

# ── 1. Gender typos ───────────────────────────────────────────────────────────
valid_gender = {'M', 'F'}
gender_typos = ~df['gender'].isin(valid_gender)
issues['Gender typos'] = gender_typos.sum()
print(f'Gender typos: {gender_typos.sum()} rows → {df.loc[gender_typos, "gender"].unique()}')

# ── 2. Weight: lbs entered as kg (>150 kg is suspicious; BMI should flag it) ──
weight_lbs_flag = (df['weight_kg'] > 150)
issues['Weight > 150 kg (possible lbs)'] = weight_lbs_flag.sum()
print(f'\nWeight > 150 kg: {weight_lbs_flag.sum()} rows (possible lb/kg confusion)')
print(df.loc[weight_lbs_flag, ['weight_kg','height_cm','bmi']].describe())

# ── 3. Temperature artifacts ──────────────────────────────────────────────────
temp_artifact = (df['temp_celsius_mean'] > 42) | (df['temp_celsius_mean'] < 32)
issues['Temperature artifacts'] = temp_artifact.sum()
print(f'\nTemp artifacts: {temp_artifact.sum()} rows')
print(df.loc[temp_artifact, 'temp_celsius_mean'].value_counts().head())

# ── 4. BP swap errors (DBP > SBP) ─────────────────────────────────────────────
bp_swap = df['dbp_mean'] > df['sbp_mean']
issues['BP swap (DBP > SBP)'] = bp_swap.sum()
print(f'\nBP swap errors (DBP > SBP): {bp_swap.sum()} rows')

# ── 5. HR double-counted ──────────────────────────────────────────────────────
hr_double = df['hr_mean'] > 180
issues['HR > 180 (possible double-count)'] = hr_double.sum()
print(f'\nHR > 180: {hr_double.sum()} rows')

# ── 6. GCS paradox ────────────────────────────────────────────────────────────
gcs_paradox = (df['gcs_total'] == 15) & (df['mechanical_ventilation'] == 1)
issues['GCS=15 but intubated'] = gcs_paradox.sum()
print(f'\nGCS=15 + intubated (documentation error): {gcs_paradox.sum()} rows')

# ── 7. WBC freetext ───────────────────────────────────────────────────────────
wbc_freetext = df['wbc'].astype(str).str.contains(r'^[>< ]', na=False)
issues['WBC freetext entries'] = wbc_freetext.sum()
print(f'\nWBC freetext (e.g. ">30"): {wbc_freetext.sum()} rows')

# ── 8. SpO2 impossibility ────────────────────────────────────────────────────
spo2_impossible = df['spo2_mean'] > 100
issues['SpO2 > 100%'] = spo2_impossible.sum()
print(f'\nSpO2 > 100: {spo2_impossible.sum()} rows')

# ── Summary table ─────────────────────────────────────────────────────────────
print('DATA QUALITY AUDIT SUMMARY')
for k, v in issues.items():
    print(f'  {k:45s}: {v:4d} rows')
print(f'  TOTAL issues detected: {sum(issues.values())}')

---
## Step 4 — Exploratory Data Analysis (EDA)

Now we understand the *clinical patterns* in the data before modeling. Key questions:
1. Which vitals and labs separate sepsis from non-sepsis?
2. How do comorbidities correlate with sepsis incidence?
3. Are multicollinear features present?
4. What does the MNAR (Missing Not At Random) pattern look like?

In [ ]:
# ── Continuous features: sepsis vs non-sepsis distributions ──────────────────
key_features = ['lactate_mmol', 'sofa_score', 'map_mean', 'hr_mean',
                'temp_celsius_mean', 'creatinine', 'wbc', 'ph_arterial',
                'platelet_count', 'pao2_fio2_ratio', 'gcs_total', 'apache_iv']

# WBC needs numeric conversion for plotting
df_plot = df.copy()
df_plot['wbc'] = pd.to_numeric(df_plot['wbc'], errors='coerce')

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    for label, color, name in [(0, '#4C72B0', 'No Sepsis'), (1, '#DD8452', 'Sepsis')]:
        subset = df_plot.loc[df_plot[TARGET] == label, feat].dropna()
        axes[i].hist(subset, bins=40, alpha=0.6, color=color, label=name, density=True)
    axes[i].set_title(feat, fontweight='bold', fontsize=10)
    axes[i].legend(fontsize=8)
    axes[i].set_ylabel('Density')

fig.suptitle('Key Feature Distributions: Sepsis vs No Sepsis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Comorbidity sepsis rates ──────────────────────────────────────────────────
comorbidities = ['diabetes','hypertension','chf','copd','chronic_kidney_disease',
                 'liver_disease','immunosuppression','cad','atrial_fibrillation','cancer_active']

sep_rates = {}
for c in comorbidities:
    yes = df.loc[df[c]==1, TARGET].mean()
    no  = df.loc[df[c]==0, TARGET].mean()
    sep_rates[c] = {'With condition': yes, 'Without condition': no}

rates_df = pd.DataFrame(sep_rates).T * 100

rates_df.plot(kind='barh', figsize=(11, 6), color=['#DD8452','#4C72B0'], edgecolor='white')
plt.axvline(df[TARGET].mean()*100, ls='--', color='black', label=f'Overall rate ({df[TARGET].mean()*100:.1f}%)')
plt.xlabel('Sepsis Rate (%)')
plt.title('Sepsis Incidence by Comorbidity', fontweight='bold', fontsize=13)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap (numeric features) ────────────────────────────────────
numeric_df = df_plot[key_features + ['sbp_mean','dbp_mean','inr','sodium','bicarbonate',
                                      'qsofa','sirs_criteria', TARGET]].copy()
corr_mat = numeric_df.corr()

mask = np.triu(np.ones_like(corr_mat, dtype=bool))
fig, ax = plt.subplots(figsize=(16, 13))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr_mat, mask=mask, annot=True, fmt='.2f', cmap=cmap,
            center=0, linewidths=0.4, ax=ax,
            annot_kws={'size': 8}, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with target
print('Top 15 features correlated with sepsis_label:')
print(corr_mat[TARGET].abs().sort_values(ascending=False).head(16)[1:])

In [ ]:
# ── MNAR pattern: missingness by sepsis label ─────────────────────────────────
lab_cols = ['wbc','creatinine','platelet_count','bilirubin_total','glucose',
            'ph_arterial','inr','sodium','potassium','chloride','bicarbonate',
            'hematocrit','hemoglobin']

miss_by_label = pd.DataFrame({
    'Sepsis=0': df.loc[df[TARGET]==0, lab_cols].isnull().mean(),
    'Sepsis=1': df.loc[df[TARGET]==1, lab_cols].isnull().mean(),
})

miss_by_label.plot(kind='bar', figsize=(13, 5), color=['#4C72B0','#DD8452'], edgecolor='white')
plt.title('Missing Lab Values by Sepsis Label (MNAR pattern)', fontweight='bold', fontsize=13)
plt.ylabel('Missing Fraction')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Label')
plt.tight_layout()
plt.show()
print('\n🔑 Key insight: Sicker patients (sepsis=1) have MORE missing labs — this is MNAR.')
print('   Simple mean imputation will be BIASED. KNN or model-based imputation is better.')

In [ ]:
# ── Age distribution & ICU admission hour patterns ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE)

for label, color, name in [(0,'#4C72B0','No Sepsis'),(1,'#DD8452','Sepsis')]:
    axes[0].hist(df.loc[df[TARGET]==label,'age'], bins=30, alpha=0.6, color=color, label=name, density=True)
axes[0].set_title('Age Distribution by Label', fontweight='bold')
axes[0].set_xlabel('Age'); axes[0].legend()

hourly = df.groupby('icu_admit_time_hour')[TARGET].agg(['mean','count']).reset_index()
axes[1].bar(hourly['icu_admit_time_hour'], hourly['count'], color='#4C72B0', alpha=0.6, label='Admissions')
ax2 = axes[1].twinx()
ax2.plot(hourly['icu_admit_time_hour'], hourly['mean']*100, 'o-', color='#DD8452', label='Sepsis Rate %', linewidth=2)
axes[1].set_title('Admissions & Sepsis Rate by Hour', fontweight='bold')
axes[1].set_xlabel('Hour of Day'); axes[1].set_ylabel('Count')
ax2.set_ylabel('Sepsis Rate (%)')
plt.tight_layout(); plt.show()

---
## Step 5 — Data Cleaning

We fix all the data quality issues identified in Step 3 using domain knowledge:

| Issue | Fix Strategy |
|-------|--------------|
| Weight in lbs | Divide by 2.205 |
| Gender typos | Map to nearest valid value |
| Temp artifacts (30/50°C) | Set to NaN → impute later |
| BP swap (DBP > SBP) | Swap values back |
| HR > 180 (double-count) | Divide by 2 |
| WBC freetext | Strip prefix, cast to float |
| SpO2 > 100 | Cap at 100 |
| GCS=15 + intubated | Flag as `gcs_documentation_error` |

In [ ]:
df_clean = df_raw.copy()

# ── 1. Gender typos ───────────────────────────────────────────────────────────
gender_map = {'M':'M','F':'F','Mael':'M','Femail':'F','Male':'M','Female':'F'}
df_clean['gender'] = df_clean['gender'].map(lambda x: gender_map.get(str(x).strip(), x))
print(f"Gender unique after fix: {df_clean['gender'].unique()}")

# ── 2. Weight: lbs → kg ───────────────────────────────────────────────────────
lbs_mask = df_clean['weight_kg'] > 150
df_clean.loc[lbs_mask, 'weight_kg'] = df_clean.loc[lbs_mask, 'weight_kg'] / 2.205
# Recompute BMI
df_clean['bmi'] = df_clean['weight_kg'] / (df_clean['height_cm'] / 100)**2
print(f'Weight fixed: {lbs_mask.sum()} rows converted lbs→kg')

# ── 3. Temperature artifacts ─────────────────────────────────────────────────
temp_cols = ['temp_celsius_mean','temp_celsius_max','temp_celsius_min']
for col in temp_cols:
    bad = (df_clean[col] > 42) | (df_clean[col] < 32)
    df_clean.loc[bad, col] = np.nan
print(f'Temp artifacts → NaN: {temp_artifact.sum()} rows')

# ── 4. BP swap errors ─────────────────────────────────────────────────────────
swap_mask = df_clean['dbp_mean'] > df_clean['sbp_mean']
df_clean.loc[swap_mask, ['sbp_mean','dbp_mean']] = \
    df_clean.loc[swap_mask, ['dbp_mean','sbp_mean']].values
print(f'BP swaps fixed: {swap_mask.sum()} rows')

# ── 5. HR double-counting ────────────────────────────────────────────────────
for col in ['hr_mean','hr_max','hr_min']:
    hr_double_mask = df_clean[col] > 180
    df_clean.loc[hr_double_mask, col] = df_clean.loc[hr_double_mask, col] / 2
print(f'HR double-counts fixed: {(df_clean["hr_mean"] > 180).sum()} remaining > 180')

# ── 6. WBC freetext ──────────────────────────────────────────────────────────
df_clean['wbc'] = (
    df_clean['wbc'].astype(str)
    .str.replace(r'^[><\s]+', '', regex=True)
    .pipe(pd.to_numeric, errors='coerce')
)
print(f'WBC dtype after fix: {df_clean["wbc"].dtype}')

# ── 7. SpO2 cap ───────────────────────────────────────────────────────────────
for col in ['spo2_mean','spo2_max','spo2_min']:
    df_clean[col] = df_clean[col].clip(upper=100)

# ── 8. Flag GCS paradox (keep original GCS, add flag) ───────────────────────
df_clean['gcs_documentation_error'] = (
    (df_clean['gcs_total'] == 15) & (df_clean['mechanical_ventilation'] == 1)
).astype(int)
print(f'GCS documentation error flag: {df_clean["gcs_documentation_error"].sum()} rows')

print('\n Data cleaning complete.')

---
## Step 6 — Feature Engineering

We create clinically meaningful derived features. These often carry more predictive power than raw measurements:

| Feature | Formula | Clinical Meaning |
|---------|---------|------------------|
| **Shock Index** | HR / SBP | >1.0 = hemodynamic instability |
| **Pulse Pressure** | SBP − DBP | Narrowing = poor perfusion |
| **MAP (recalculated)** | (2×DBP + SBP)/3 | True perfusion pressure |
| **Age × Creatinine** | age × creatinine | Renal risk interaction |
| **Lactate × MAP** | lactate × (1/MAP) | Tissue hypoperfusion composite |
| **Renal Risk** | age + 5×creatinine | CKD progression risk |
| **Missing Lab Count** | Sum of null flags | Severity proxy (MNAR) |
| **Comorbidity Burden** | Sum of 10 binary flags | Overall frailty score |

In [ ]:
df_feat = df_clean.copy()

# ── Vital-sign derived features ───────────────────────────────────────────────
df_feat['shock_index']    = df_feat['hr_mean'] / df_feat['sbp_mean'].replace(0, np.nan)
df_feat['pulse_pressure'] = df_feat['sbp_mean'] - df_feat['dbp_mean']
df_feat['map_recalc']     = (2*df_feat['dbp_mean'] + df_feat['sbp_mean']) / 3
df_feat['hr_sbp_ratio']   = df_feat['hr_mean'] / df_feat['sbp_mean'].replace(0, np.nan)
df_feat['spo2_rr_ratio']  = df_feat['spo2_mean'] / df_feat['respiratory_rate_mean'].replace(0, np.nan)

# ── Lab interaction features ──────────────────────────────────────────────────
df_feat['lactate_map_interaction'] = df_feat['lactate_mmol'] / df_feat['map_mean'].replace(0, np.nan)
df_feat['age_creatinine']          = df_feat['age'] * df_feat['creatinine'].fillna(df_feat['creatinine'].median())
df_feat['renal_risk']              = df_feat['age'] + 5 * df_feat['creatinine'].fillna(df_feat['creatinine'].median())
df_feat['inflammation_score']      = df_feat['wbc'].fillna(df_feat['wbc'].median()) * df_feat['temp_celsius_mean'].fillna(37)

# ── MNAR proxy: count of missing labs per patient ────────────────────────────
lab_cols_all = ['wbc','lactate_mmol','creatinine','platelet_count','bilirubin_total',
                'glucose','ph_arterial','inr','sodium','potassium','chloride',
                'bicarbonate','hematocrit','hemoglobin']
df_feat['missing_lab_count'] = df_feat[lab_cols_all].isnull().sum(axis=1)

# ── Comorbidity burden ────────────────────────────────────────────────────────
comorbidity_cols = ['diabetes','hypertension','chf','copd','chronic_kidney_disease',
                    'liver_disease','immunosuppression','cad','atrial_fibrillation','cancer_active']
df_feat['comorbidity_burden'] = df_feat[comorbidity_cols].sum(axis=1)

# ── BMI categories ────────────────────────────────────────────────────────────
df_feat['bmi_category'] = pd.cut(
    df_feat['bmi'],
    bins=[0, 18.5, 25, 30, 35, 100],
    labels=['Underweight','Normal','Overweight','Obese1','Obese2']
).astype(str)

# ── Age groups ───────────────────────────────────────────────────────────────
df_feat['age_group'] = pd.cut(
    df_feat['age'],
    bins=[0, 40, 60, 75, 100],
    labels=['<40','40-60','60-75','75+']
).astype(str)

new_features = ['shock_index','pulse_pressure','map_recalc','lactate_map_interaction',
                'age_creatinine','renal_risk','missing_lab_count','comorbidity_burden']
print('New features created:')
print(df_feat[new_features + [TARGET]].describe().T[['mean','std','min','max']])

In [ ]:
# ── Feature importance preview: correlation with target ───────────────────────
num_cols_all = df_feat.select_dtypes(include=[np.number]).columns.tolist()
corr_with_target = df_feat[num_cols_all].corr()[TARGET].drop(TARGET)
top_pos = corr_with_target.sort_values(ascending=False).head(15)
top_neg = corr_with_target.sort_values(ascending=True).head(5)
top_combined = pd.concat([top_pos, top_neg]).sort_values()

colors = ['#DD8452' if v > 0 else '#4C72B0' for v in top_combined.values]
top_combined.plot(kind='barh', color=colors, figsize=(11,7), edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Sepsis Label', fontweight='bold', fontsize=13)
plt.xlabel('Pearson r')
plt.tight_layout()
plt.show()

---
## Step 7 — Preprocessing Pipeline

We build a **sklearn `ColumnTransformer` pipeline** that:
1. Separates numerical and categorical columns
2. Imputes missing values (median for numeric, constant for categorical)
3. Scales numeric features with `StandardScaler`
4. One-hot encodes categorical features

This pipeline is fitted **only on training data** to prevent data leakage.

In [ ]:
# ── Prepare feature matrix ────────────────────────────────────────────────────
drop_from_features = DROP_COLS + [TARGET,
    # Remove raw MAP since we recalculated it; remove sedation_score (high missingness, ventilated only)
    'sedation_score'
]

feature_df = df_feat.drop(columns=drop_from_features, errors='ignore')

# Identify column types
categorical_cols = feature_df.select_dtypes(include=['object','category']).columns.tolist()
numerical_cols   = feature_df.select_dtypes(include=[np.number]).columns.tolist()

print(f'Categorical features: {len(categorical_cols)}')
print(f'{categorical_cols}')
print(f'\nNumerical features: {len(numerical_cols)}')
print(f'Total features before preprocessing: {len(categorical_cols) + len(numerical_cols)}')

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# ── Train/test split ──────────────────────────────────────────────────────────
X = feature_df.copy()
y = df_feat[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape} | Sepsis rate: {y_train.mean():.3f}')
print(f'Test:  {X_test.shape}  | Sepsis rate: {y_test.mean():.3f}')

# ── Preprocessing transformers ────────────────────────────────────────────────
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='drop'
)

# Fit on train, transform both
X_train_pp = preprocessor.fit_transform(X_train)
X_test_pp  = preprocessor.transform(X_test)

# Get feature names after OHE
ohe_names = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_cols)
feature_names_pp = numerical_cols + list(ohe_names)

print(f'\nFeature matrix after preprocessing: {X_train_pp.shape}')
print(f'Numeric: {len(numerical_cols)} | OHE-expanded categoricals: {len(ohe_names)}')

---
## Step 8 — Model Training

We train XGBoost model's which has a strengths High performance and handles missing. but on the other hand there are weaknes, namely Needs tuning and can overfit

In [ ]:
# define model
xgb_model = xgb.XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),
        use_label_encoder=False, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )

# ── Train models & collect results ───────────────────────────────────────
xgb_model.fit(X_train_pp, y_train)

# 3. Lakukan Prediksi pada data uji
y_prob = xgb_model.predict_proba(X_test_pp)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print('\nModels trained.')

---
## Step 9 — Evaluation & SHAP

For clinical applications, **ROC-AUC alone is insufficient** — the class is imbalanced.
We use:
- **PR-AUC (Precision-Recall AUC)**: Best metric for imbalanced binary classification
- **ROC-AUC**: Overall discrimination ability
- **F1 Score**: Balance between precision and recall at default 0.5 threshold
- **Brier Score**: Calibration — how well-calibrated are the predicted probabilities?
- **SHAP**: Model explainability

In [ ]:
print("HASIL EVALUASI XGBOOST\n")

# 1. Displaying Accuracy & Recall Scores
roc_auc = roc_auc_score(y_test, y_prob)
print(f"Skor ROC-AUC : {roc_auc:.4f}")
print("\nLaporan Detail (Classification Report):")
print(classification_report(y_test, y_pred))

# 2. SHAP Analysis (Dissecting AI Logic)
print("\nMenghitung SHAP Values untuk transparansi (Explainability)...")
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_pp)

# Showing a graph of which features are the most triggers for Sepsis!
shap.summary_plot(shap_values, X_test_pp, feature_names=feature_names_pp)

---
## Step 10 — Threshold Tuning (Clinical Optimization)

In clinical settings, **false negatives are more dangerous** than false positives:
- Missing sepsis (FN) → patient deteriorates, potentially fatal
- Over-treatment (FP) → unnecessary antibiotics, resource waste

We find the optimal decision threshold by maximizing a **clinical F-beta score** (β=2 weights recall 2× over precision).

In [ ]:
thresholds = np.arange(0.05, 0.80, 0.01)
threshold_results = []

# Testing various thresholds to minimize undetected Sepsis patients (False Negative)
for t in thresholds:
    preds = (y_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f2        = (1 + 4) * precision * recall / (4*precision + recall + 1e-9)
    threshold_results.append({
        'threshold': t, 'recall': recall, 'f2_score': f2, 'fn': fn
    })

thr_df = pd.DataFrame(threshold_results)
best_thr_row = thr_df.loc[thr_df['f2_score'].idxmax()]
optimal_threshold = best_thr_row['threshold']

print(f'\nThreshold Optimal (memaksimalkan deteksi Sepsis): {optimal_threshold:.2f}')
print(f'   Recall (Sensitivitas):    {best_thr_row["recall"]:.3f}')
print(f'   Pasien Sepsis yang terlewat (False Negatives): {int(best_thr_row["fn"])}')

y_pred_optimal = (y_prob >= optimal_threshold).astype(int)

---
## Step 11 — Final Data Summary

### Key Findings

In [ ]:
# ── Save best model predictions for submission ────────────────────────────────
submission = pd.DataFrame({
    'subject_id': df_feat.iloc[X_test.index.tolist() if hasattr(X_test,'index') else range(len(y_test))]['subject_id'].values,
    'sepsis_probability': y_prob,
    'sepsis_prediction_default': y_pred,
    'sepsis_prediction_optimal': y_pred_optimal,
    'true_label': y_test,
})

submission.to_csv('sepsis_predictions_MVP.csv', index=False)
print(f'Predictions saved: sepsis_predictions.csv ({len(submission)} rows)')

files.download('sepsis_predictions_MVP.csv')
print("currently downloading files")

print(submission.head())

---
## Step 12 — Export to .pkl

### For reading python file

In [ ]:
import joblib

nama_model = 'sepsis_xgboost_model.pkl'
joblib.dump(xgb_model, nama_model)

nama_preprocessor = 'sepsis_preprocessor.pkl'
joblib.dump(preprocessor, nama_preprocessor)

import json
with open('fitur_input.json', 'w') as f:
    json.dump(list(X_train.columns), f)

print(f"Model berhasil dibungkus!")
files.download(nama_model)
files.download(nama_preprocessor)
files.download('fitur_input.json')